# Notebook 02: the classic detectors on KDD99

**Class 03_01.** Ten detectors, one protocol (notebook 01): fit on normal training rows, threshold at 1% false
alarms on normal validation rows, test with 1.8% attacks. Every detector returns a score where higher = more anomalous.

| family | detector | score |
|:--|:--|:--|
| statistics | z-score, diagonal Gaussian, Mahalanobis | $\max_j\lvert z_j\rvert$, $\sum_j z_j^2$, $d_M^2$ |
| clustering | k-means (k = 20), Gaussian mixture (10 components) | distance to the nearest centre, $-\log p(\mathbf{x})$ |
| neighbours | kNN distance (k = 5), LOF (k = 20) | distance to the 5th neighbour, local density ratio |
| isolation | Isolation Forest (200 trees) | $2^{-E[h]/c(n)}$ |
| boundary | one-class SVM ($\nu=0.01$, RBF) | $-$ signed distance to the boundary |
| subspace | PCA (95% of the variance) | squared reconstruction error |

In [ ]:
import sys
import time

import numpy as np
import polars as pl

sys.path[:0] = [".", "../../src"]  # this folder and the generic libraries of the repository
import aasplot as ap
import anomlib as an
from aasplot import BLUE, GOLD, GREEN, GREY, LIGHT, RED

an.table_config()
ap.setup("03")
df = an.load_kdd()
sp = an.kdd_split(df, seed=0)
Xtr, Xva, Xte, y = sp.X("train"), sp.X("val"), sp.X("test"), sp.y_test
cats = sp.test["category"]

## A. One split, all detectors

In [ ]:
rows, scores = [], {}
for det in an.zoo(seed=0):
    t0 = time.time()
    det.fit(Xtr)
    s_va, s_te = det.score(Xva), det.score(Xte)
    scores[det.name] = (s_va, s_te)
    rows.append({"detector": det.name, **an.evaluate(s_te, y, s_va), **an.recall_by_category(s_te, cats, s_va),
                 "seconds": time.time() - t0})  # fmt: skip
res = pl.DataFrame(rows)
print(res.sort("AUPRC", descending=True))

In [ ]:
GMM = "GMM (-log-likelihood)"


SHORT = {
    GMM: "GMM",
    "z-score (max |z|)": "z-score",
    "Gaussian (diagonal)": "diag. Gaussian",
    "k-means distance": "k-means",
    "PCA reconstruction": "PCA recon.",
}


def ten_detectors(rows) -> None:
    items = sorted((r["AUPRC"], r["detector"]) for r in rows)
    f, ax = ap.fig("full", 1.6)
    ax.barh([SHORT.get(k, k) for _, k in items], [v for v, _ in items], color=BLUE, height=0.65)
    for i, (v, _) in enumerate(items):
        ax.text(v + 0.008, i, f"{v:.3f}", va="center", fontsize=7)
    ax.set(xlabel="AUPRC (1.8% attacks; a random score gets 0.018)", xlim=(0, 0.78))
    ax.grid(axis="y", visible=False)
    ap.save(f, "ten_detectors")


ten_detectors(res.to_dicts())

In [ ]:
CATS = an.CATEGORIES


def by(rows, key: str = "detector") -> dict:
    return {r[key]: r for r in rows}


PAL = [BLUE, RED, GOLD, GREEN]


def grouped(ax, groups, series, ylabel="", colors=PAL, width=0.8):
    n = len(series)
    for i, (name, vals) in enumerate(series):
        pos = np.arange(len(groups)) + (i - (n - 1) / 2) * width / n
        bars = ax.bar(pos, vals, width / n, color=colors[i], label=name, lw=0)
        for b, v in zip(bars, vals):
            ax.text(
                b.get_x() + b.get_width() / 2,
                v + 0.015,
                f"{v:.2f}".lstrip("0") if v < 1 else "1",
                ha="center",
                fontsize=6.5,
            )
    ax.set_xticks(np.arange(len(groups)), groups)
    ax.set(ylabel=ylabel, ylim=(0, 1.12))
    ax.grid(axis="x", visible=False)


def recall_cat(rows) -> None:
    d = by(rows)
    names = [GMM, "PCA reconstruction", "Isolation Forest", "LOF"]
    f, ax = ap.fig("full", 1.35)
    grouped(ax, CATS, [(SHORT.get(n, n), [d[n][c] for c in CATS]) for n in names], "recall at 1% false alarms")
    ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.2))
    ap.save(f, "recall_cat")


recall_cat(res.to_dicts())

**Reading the table.**
* Every detector has AUROC above 0.94, but AUPRC between 0.33 and 0.68: the base rate at work.
* **dos and probe** are easy (volume and error-rate features move far from normal); **r2l** is hard for all of them:
  a password-guessing or warez connection looks like an ordinary login. It differs in content, not in shape.
* **LOF misses dos** (recall 0.08): the floods are *dense* clusters, so their local density ratio is about 1
  (notebook 00, section D). The Isolation Forest misses **u2r**: a few unusual content counts are hard to isolate
  with random axis cuts when most features are ordinary.
* The simple ones are competitive: the Mahalanobis distance, a covariance and a matrix product, ranks in the middle;
  PCA reconstruction has the best F1. This matches large benchmarks: no unsupervised detector wins everywhere
  (ADBench, Han et al. 2022).

## B. Is the ranking stable? Three splits

In [ ]:
rows = []
for seed in [0, 1, 2]:
    s = an.kdd_split(df, seed=seed)
    a, b, c, yy = s.X("train"), s.X("val"), s.X("test"), s.y_test
    for det in an.zoo(seed=seed):
        det.fit(a)
        rows.append({"seed": seed, "detector": det.name, **an.evaluate(det.score(c), yy, det.score(b))})
seeds = pl.DataFrame(rows)

In [ ]:
def splits(rows) -> None:
    """rows: one dict per (seed, detector) with AUPRC."""
    acc: dict = {}
    for r in rows:
        acc.setdefault(r["detector"], []).append(r["AUPRC"])
    keep = {GMM, "One-class SVM", "kNN distance", "Isolation Forest", "PCA reconstruction", "Mahalanobis"}
    data = sorted(((np.mean(v), np.std(v, ddof=1), k) for k, v in acc.items() if k in keep), reverse=True)
    f, ax = ap.fig("full", 1.35)
    ys = np.arange(len(data))[::-1]
    ax.barh(
        ys,
        [m for m, _, _ in data],
        xerr=[s for _, s, _ in data],
        color=BLUE,
        height=0.6,
        error_kw={"ecolor": RED, "lw": 1.1, "capsize": 2},
    )
    ax.set_yticks(ys, [SHORT.get(k, k) for _, _, k in data])
    ax.set(xlabel="AUPRC, mean ± std over 3 splits", xlim=(0.4, 0.75))
    ax.grid(axis="y", visible=False)
    ap.save(f, "splits")


splits(rows)

In [ ]:
print(seeds.group_by("detector").agg(pl.col("AUPRC").mean().alias("AUPRC mean"), pl.col("AUPRC").std().alias("AUPRC std"),
                                     pl.col("F1").mean().alias("F1 mean"), pl.col("F1").std().alias("F1 std"))
      .sort("AUPRC mean", descending=True))  # fmt: skip

With only 326 attacks in a test set, differences of a few AUPRC points between detectors are within the spread
across splits. **Never rank detectors on one split.**

## C. Unsupervised setting: attacks in the training data

In practice the "normal" training data are not clean. Add a share of attacks from the pool to the training rows.

In [ ]:
rng = np.random.default_rng(0)
Xp = sp.X("pool")
rows = []
for rate in [0.0, 0.01, 0.05]:
    k = int(rate * len(Xtr))
    Xc = np.vstack([Xtr, Xp[rng.choice(len(Xp), k, replace=False)]])
    for det in [an.Mahalanobis(), an.PCARecon(), an.KNNDist(), an.OCSVM(), an.IForest()]:
        det.fit(Xc)
        rows.append(
            {
                "contamination": rate,
                "detector": det.name,
                "AUPRC": an.evaluate(det.score(Xte), y, det.score(Xva))["AUPRC"],
            }
        )
cont = pl.DataFrame(rows).pivot(on="contamination", index="detector", values="AUPRC")
print(cont)

In [ ]:
def contam(rows) -> None:
    """rows: one dict per (contamination rate, detector) with AUPRC; rates 0, 0.01, 0.05."""
    f, ax = ap.fig("full", 1.55)
    dets = list(dict.fromkeys(r["detector"] for r in rows))
    for name, col, mk in zip(dets, (BLUE, GREEN, GOLD, GREY, RED), "sD^os"):
        pts = sorted((100 * r["contamination"], r["AUPRC"]) for r in rows if r["detector"] == name)
        ax.plot(*zip(*pts), color=col, marker=mk, ms=3.5, label=name, lw=1.5 if name == "Isolation Forest" else 1.1)
    ax.set(xlabel="attacks mixed into the training rows (%)", ylabel="AUPRC", xticks=[0, 1, 5], ylim=(0, 0.72))
    ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
    ap.save(f, "contam")


contam(rows)

1% of attacks (mostly DoS from the pool) in training halves the AUPRC of the covariance, PCA, kNN and SVM detectors:
the floods become "normal". The Isolation Forest barely moves: each tree sees a random subsample of 256 rows, and a
small contamination rarely changes the shape of the trees (Liu et al. 2008 designed it for this setting).

## D. What the score explains

A detector that says only "anomalous" is hard to triage. For the diagonal Gaussian, the score is a sum over features,
so each feature's share is its explanation; for PCA, the per-feature reconstruction error.

In [ ]:
dg = an.DiagGauss().fit(Xtr)
names = np.array(sp.prep.names)
for cat in an.CATEGORIES:
    x = Xte[(cats == cat).to_numpy()]
    contrib = (((x - dg.mu) / dg.sd) ** 2).mean(0)
    top = np.argsort(contrib)[::-1][:4]
    print(f"{cat:6}", ", ".join(f"{names[i]} ({contrib[i]:.0f})" for i in top))

## D. Blind source separation: PCA, ICA and NMF

PCA is one decomposition of an event into a few sources; **ICA** looks for statistically independent sources and
**NMF** for non-negative parts. All three score an event by the squared residual of its reconstruction (ICA also by
its largest source). NMF comes from `pyNNMF` (HALS); a new row gets its non-negative weights with the parts fixed.
ICA with $k$ components spans the same subspace as PCA with $k$, so its residual equals PCA's.

In [ ]:
k = int(an.PCARecon().fit(Xtr).p.n_components_)
rows = []
for det in [an.PCARecon(), an.ICARecon(k, "recon"), an.ICARecon(k, "source"), an.NMFRecon(k)]:
    t0 = time.time()
    det.fit(Xtr)
    s_va, s_te = det.score(Xva), det.score(Xte)
    rows.append({"detector": det.name, **an.evaluate(s_te, y, s_va), **an.recall_by_category(s_te, cats, s_va), "seconds": time.time() - t0})  # fmt: skip
print(f"k = {k} components (95% of the variance)")
print(pl.DataFrame(rows).select(["detector", "AUROC", "AUPRC", *an.CATEGORIES, "seconds"]))

In [ ]:
def bss_results(rows) -> None:
    """rows: PCA, ICA (recon), ICA (source), NMF reconstruction with AUPRC and the recall per category."""
    d = by(rows)
    order = [
        ("PCA reconstruction", "PCA"),
        ("ICA (recon)", "ICA, residual"),
        ("ICA (source)", "ICA, largest source"),
        ("NMF reconstruction", "NMF, residual"),
    ]
    f, axs = ap.fig("full", 1.3, ncols=2, gridspec_kw={"wspace": 0.3, "width_ratios": [1, 1.6]})
    ax = axs[0]
    vals = [d[k]["AUPRC"] for k, _ in order]
    ax.barh(range(4)[::-1], vals, color=[BLUE, BLUE, LIGHT, GREEN], height=0.6)
    ax.set_yticks(range(4)[::-1], [n for _, n in order])
    for i, v in zip(range(4)[::-1], vals):
        ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=7)
    ax.set(xlabel="AUPRC", xlim=(0, 0.72))
    ax.grid(axis="y", visible=False)
    grouped(
        axs[1],
        CATS,
        [(n, [d[k][c] for c in CATS]) for (k, n) in (order[0], order[2], order[3])],
        "recall",
        colors=[BLUE, LIGHT, GREEN],
    )
    axs[1].legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.2), fontsize=6.5)
    ap.save(f, "bss_results")


bss_results(rows)

PCA and the ICA residual are the same detector (AUPRC 0.549); the largest-source score of ICA is worse (0.367); NMF
(0.518) misses probe and r2l: normal traffic is not a sum of non-negative parts. Every figure of this class is drawn in
a cell of notebooks 00-03 and stored in `assets/figures/plots/03/` by the same call.